In [1]:
import numpy as np
import healpy as hp


In [36]:
ra = np.array([0,30.,60,90, 90, 90])
dec = np.array([0,-10,-20,-90, -10, -20])

In [37]:
nside = 32

In [38]:
hpid, weights = hp.get_interp_weights(nside, ra, dec, lonlat=True)

In [39]:
hpid

array([[ 6207,  7114,  8148, 12284,  7135,  8159],
       [ 6080,  7115,  8149, 12285,  7136,  8160],
       [ 6208,  7242,  8277, 12286,  7264,  8288],
       [ 6209,  7243,  8278, 12287,  7265,  8289]])

In [40]:
weights

array([[0.5       , 0.554424  , 0.09733634, 0.25      , 0.3326544 ,
        0.29200901],
       [0.5       , 0.1108848 , 0.48668169, 0.25      , 0.3326544 ,
        0.29200901],
       [0.        , 0.11156373, 0.27732132, 0.25      , 0.3346912 ,
        0.41598198],
       [0.        , 0.22312746, 0.13866066, 0.25      , 0.        ,
        0.        ]])

In [42]:
np.max(weights, axis=0)

array([0.5       , 0.554424  , 0.48668169, 0.25      , 0.3346912 ,
       0.41598198])

In [43]:
m_weight = (np.max(weights, axis=0)* np.ones(weights.shape))

In [44]:
m_weight

array([[0.5       , 0.554424  , 0.48668169, 0.25      , 0.3346912 ,
        0.41598198],
       [0.5       , 0.554424  , 0.48668169, 0.25      , 0.3346912 ,
        0.41598198],
       [0.5       , 0.554424  , 0.48668169, 0.25      , 0.3346912 ,
        0.41598198],
       [0.5       , 0.554424  , 0.48668169, 0.25      , 0.3346912 ,
        0.41598198]])

In [24]:
np.where(weights < m_weight)

(array([0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3]),
 array([0, 2, 3, 1, 2, 3, 0, 1, 3, 0, 1, 2]))

In [25]:
final_weights = np.ones(weights.shape)
final_weights[np.where(weights < m_weight)] = 0

In [30]:
final_weights = final_weights / np.sum(final_weights, axis=1)

In [31]:
final_weights

array([[0., 1., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]])

In [45]:
def ra_dec_nearest_interp(nside, in_array, ra, dec):
    """Interpolate using nearest neighbor for a given RA,dec.
    Averages results if more than one HEALpix center is equally close.

    Parameters
    ----------
    nside : `int`
        HEALpix nside
    in_array : `np.NDarray`
        A valid HEALpix array
    ra : `float`
        RA value(s) to interpolate in_array to using nearest neighbor.
    dec : `float`
        Dec value(s) to interpolate in_array to using nearest neighbor.
    """

    # returns the 4 closest 
    hpid, weights = hp.get_interp_weights(nside, ra, dec, lonlat=True)

    # Find the max weight for each RA,dec
    m_weight = np.max(weights, axis=0) * np.ones(weights.shape)

    final_weights = np.ones(weights.shape)
    # Anything less than the max, set to zero weight
    final_weights[np.where(weights < m_weight)] = 0
    # Normalize in case there was more than one at the max value
    final_weights = final_weights / np.sum(final_weights, axis=0)

    result = np.sum(in_array[hpid] * final_weights, axis=0)
    return result


In [49]:
ra = np.array([270, 180, 0,30.,60,90, 90, 90])
dec = np.array([20,10,0,-10,-20,-90, -10, -20])

In [50]:
in_array = np.arange(hp.nside2npix(32))

In [51]:
ra_dec_nearest_interp(nside, in_array, ra, dec)

array([ 4000. ,  4992. ,  6143.5,  7114. ,  8149. , 12285.5,  7264. ,
        8288. ])